In [ ]:

from markovstates.factor_analysis import X, loadings, fa
import numpy as np
import pandas as pd

n_samples, n_features = X.shape
ratio = n_features / n_samples

# Marchenko-Pastur upper bound (assuming σ²=1 after scaling)

# Rough autocorrelation correction
autocorr = pd.DataFrame(X).apply(lambda col: col.autocorr(lag=1)).mean()
n_eff = n_samples * (1 - autocorr) / (1 + autocorr)

lambda_max_corrected = (1 + np.sqrt(n_features / n_eff)) ** 2

# Get eigenvalues of the correlation matrix
eigenvalues = np.linalg.eigvalsh(np.corrcoef(X.T))
eigenvalues = np.sort(eigenvalues)[::-1]

# Count how many are above the noise threshold
n_signal = np.sum(eigenvalues > lambda_max_corrected)
print(f"Signal components: {n_signal}")
print(f"Noise threshold: {lambda_max_corrected:.3f}")
print(eigenvalues)

FINAL_FEATURES = [
    "apparent_temperature",   # Factor 1 anchor
    "wind_speed_10m",         # Factor 2 anchor
    "dew_point_2m",           # Factor 3 anchor
]

# ok from our RMT assumptions, we see that just that alone suggests to only use 
# 2 components for FA, but interpreting the physical data, we find that underlying 
# factor 3 is probably 

                      Factor 1  Factor 2  Factor 3
temperature_2m            0.32      0.72      0.28
relative_humidity_2m     -0.64     -0.77     -0.00
dew_point_2m             -0.55     -0.38      0.35
precipitation            -0.19     -0.14      0.06
surface_pressure          0.21      0.13     -0.25
cloud_cover              -0.15     -0.13      0.14
cloud_cover_mid          -0.20     -0.21      0.26
wind_speed_10m            0.92     -0.29      0.01
wind_speed_100m           0.91     -0.41      0.01
wind_direction_10m       -0.31      0.03      0.80
wind_direction_100m      -0.28      0.06      0.85
direct_radiation          0.16      0.61     -0.06
Signal components: 2
Noise threshold: 2.114
[3.67537397 2.14971946 1.69526592 1.25599345 0.85436748 0.73596921
 0.64547576 0.41609842 0.32252193 0.19975825 0.02844494 0.02101121]


In [15]:
# looking at the factor analysis results, it seems that the temperature_2m and apparent_temperature variables 
# were correlating to each other and giving them an overestimated factor strength;  we shall remove one of them and 
# run the code again
import numpy as np
import pandas as pd
from markovstates.data_collect import hourly_dataframe

onetempdf = hourly_dataframe.drop(['apparent_temperature', 'date'], axis = 1)
feat_cols = onetempdf.columns

from sklearn.preprocessing import StandardScaler
sclr = StandardScaler()
sclr.fit(onetempdf[feat_cols])
normalized_df = sclr.transform(onetempdf[feat_cols]).astype(np.float64)

print(normalized_df)

# factor analysis done again:

from sklearn.decomposition import FactorAnalysis
fa = FactorAnalysis(n_components=3)
fa.fit(normalized_df)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feat_cols,
    columns=["Factor 1", "Factor 2", "Factor 3"]
)
print(loadings.round(2))

# yep, much better readings now

[[-1.55685008  1.78473032  0.26960427 ...  0.69341469  0.76852602
  -0.6806128 ]
 [-1.58526838  1.79956925  0.24263503 ...  0.87798697  1.01809466
  -0.6806128 ]
 [-1.54737723  1.69442225  0.20218126 ...  1.12246227  1.21313131
  -0.6806128 ]
 ...
 [ 1.75861466 -1.38381422 -0.43159506 ...  0.49948087  0.53833157
  -0.6806128 ]
 [ 1.47443187 -1.25572252 -0.37765664 ...  0.4558568   0.50719005
  -0.6806128 ]
 [ 1.17130399 -1.0510112  -0.1888722  ...  0.27103907  0.47454849
  -0.6806128 ]]
                      Factor 1  Factor 2  Factor 3
temperature_2m            0.32      0.72      0.28
relative_humidity_2m     -0.64     -0.77     -0.00
dew_point_2m             -0.55     -0.38      0.35
precipitation            -0.19     -0.14      0.06
surface_pressure          0.21      0.13     -0.25
cloud_cover              -0.15     -0.13      0.14
cloud_cover_mid          -0.20     -0.21      0.26
wind_speed_10m            0.92     -0.29      0.01
wind_speed_100m           0.91     -0.41      0.0

In [16]:
from sklearn.decomposition import FactorAnalysis
import pandas as pd
import numpy as np
from markovstates.preprocessing import fit_scaler, apply_scaler, extract_features
from markovstates.data_collect import hourly_dataframe

# constructing variables

feature_cols = extract_features(hourly_dataframe)
scaler = fit_scaler(hourly_dataframe, feature_cols)
feature_mat = apply_scaler(hourly_dataframe, feature_cols, scaler)
X = feature_mat

fa = FactorAnalysis(n_components=3)
fa.fit(X)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feature_cols,
    columns=["Factor 1", "Factor 2", "Factor 3"]
)
print(loadings.round(2))

# after factor analysis, we find that the vars contributing the most real variance are the ones below:

FINAL_FEATURES = [
    "apparent_temperature",   # Factor 1 anchor
    "wind_speed_10m",         # Factor 2 anchor
    "dew_point_2m",           # Factor 3 anchor
]

# But wait:

'''
After further analysis, we find in the factor_explore notebook that 
the variables apparent_temperature, and temperature_2m were correlating to each
other since they were more or less the same thing, strengthening each other's signal 
too much (and thus messing with other variables' influence on the factors as well)

We shall remove apparent_temperature from the df, and review the FA results again
'''


ImportError: cannot import name 'fit_scaler' from 'markovstates.preprocessing' (/Users/philipalexopoulos/markovstates/markovstates/preprocessing.py)